# Module 02: Transfer Entropy, Delay Reconstruction & Network Inference


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.special import digamma
import ipywidgets as widgets

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

def ksg_conditional_mi(x, y, z, k=5, noise_jitter=1e-8, rng=None):
    if rng is None:
        rng = np.random.default_rng(42)
    n_samples = len(x)
    x_2d = np.atleast_2d(x).reshape(n_samples, -1)
    y_2d = np.atleast_2d(y).reshape(n_samples, -1)
    z_2d = np.atleast_2d(z).reshape(n_samples, -1)
    x_std = (x_2d - np.mean(x_2d, axis=0)) / (np.std(x_2d, axis=0) + 1e-12) + rng.normal(0, noise_jitter, size=x_2d.shape)
    y_std = (y_2d - np.mean(y_2d, axis=0)) / (np.std(y_2d, axis=0) + 1e-12) + rng.normal(0, noise_jitter, size=y_2d.shape)
    z_std = (z_2d - np.mean(z_2d, axis=0)) / (np.std(z_2d, axis=0) + 1e-12) + rng.normal(0, noise_jitter, size=z_2d.shape)
    xyz = np.hstack([x_std, y_std, z_std])
    xz = np.hstack([x_std, z_std])
    yz = np.hstack([y_std, z_std])
    tree_xyz = cKDTree(xyz)
    tree_xz = cKDTree(xz)
    tree_yz = cKDTree(yz)
    tree_z = cKDTree(z_std)
    dists, _ = tree_xyz.query(xyz, k=k + 1, p=np.inf)
    eps = dists[:, k] - 1e-15
    n_xz_plus_1 = np.array([len(tree_xz.query_ball_point(xz[i], r=eps[i], p=np.inf)) for i in range(n_samples)])
    n_yz_plus_1 = np.array([len(tree_yz.query_ball_point(yz[i], r=eps[i], p=np.inf)) for i in range(n_samples)])
    n_z_plus_1 = np.array([len(tree_z.query_ball_point(z_std[i], r=eps[i], p=np.inf)) for i in range(n_samples)])
    cmi_nats = digamma(k) - np.mean(digamma(n_xz_plus_1) + digamma(n_yz_plus_1) - digamma(n_z_plus_1))
    return max(0.0, float(cmi_nats / np.log(2.0)))

def compute_transfer_entropy(source, target, conditioning=None, lag=1, k=5, rng=None):
    max_lag = max(lag, 1)
    t_future = target[max_lag:]
    t_past = target[max_lag - 1 : -1]
    s_past = source[max_lag - lag : len(source) - lag]
    if conditioning is None:
        cond_stack = t_past.reshape(-1, 1)
    else:
        c_past = conditioning[max_lag - 1 : -1]
        cond_stack = np.column_stack([t_past, c_past])
    return ksg_conditional_mi(s_past, t_future, cond_stack, k=k, rng=rng)

def generate_synthetic_motif(motif_type='Common_Driver', n_steps=600, coupling=0.65, nonlinear=True, seed=42):
    rng = np.random.default_rng(seed)
    burn_in = 100
    total_n = n_steps + burn_in
    X, Y, Z, U_hidden = [np.zeros(total_n) for _ in range(4)]
    noise_x = rng.normal(0, 0.45, total_n)
    noise_y = rng.normal(0, 0.45, total_n)
    noise_z = rng.normal(0, 0.45, total_n)
    noise_u = rng.normal(0, 0.60, total_n)
    f = (lambda val: np.tanh(1.8 * val)) if nonlinear else (lambda val: val)
    for t in range(3, total_n):
        if motif_type == 'Direct_Coupling':
            X[t] = 0.5 * X[t - 1] + noise_x[t]
            Y[t] = 0.4 * Y[t - 1] + coupling * f(X[t - 2]) + noise_y[t]
            Z[t] = 0.5 * Z[t - 1] + noise_z[t]
        elif motif_type == 'Indirect_Cascade':
            X[t] = 0.5 * X[t - 1] + noise_x[t]
            Z[t] = 0.4 * Z[t - 1] + coupling * f(X[t - 1]) + noise_z[t]
            Y[t] = 0.4 * Y[t - 1] + coupling * f(Z[t - 1]) + noise_y[t]
        elif motif_type == 'Common_Driver':
            Z[t] = 0.6 * Z[t - 1] + noise_z[t]
            X[t] = 0.3 * X[t - 1] + coupling * f(Z[t - 1]) + noise_x[t]
            Y[t] = 0.3 * Y[t - 1] + coupling * f(Z[t - 2]) + noise_y[t]
        elif motif_type == 'Unobserved_Confounder':
            U_hidden[t] = 0.6 * U_hidden[t - 1] + noise_u[t]
            X[t] = 0.3 * X[t - 1] + coupling * f(U_hidden[t - 1]) + noise_x[t]
            Y[t] = 0.3 * Y[t - 1] + coupling * f(U_hidden[t - 2]) + noise_y[t]
            Z[t] = 0.5 * Z[t - 1] + noise_z[t]
    return X[burn_in:], Y[burn_in:], Z[burn_in:]

def surrogate_significance_test(source, target, conditioning=None, lag=1, n_surrogates=25, k=5, seed=42):
    rng = np.random.default_rng(seed)
    te_emp = compute_transfer_entropy(source, target, conditioning, lag=lag, k=k, rng=rng)
    surr_vals = np.zeros(n_surrogates)
    n = len(source)
    for i in range(n_surrogates):
        shift = rng.integers(low=int(0.15 * n), high=int(0.85 * n))
        surr_vals[i] = compute_transfer_entropy(np.roll(source, shift), target, conditioning, lag=lag, k=k, rng=rng)
    p_val = (np.sum(surr_vals >= te_emp) + 1.0) / (n_surrogates + 1.0)
    return te_emp, float(np.percentile(surr_vals, 95)), p_val

def run_te_laboratory(motif_type='Common_Driver', coupling=0.70, n_samples=450):
    X, Y, Z = generate_synthetic_motif(motif_type, n_steps=n_samples, coupling=coupling, seed=42)
    rng = np.random.default_rng(42)
    lags = [1, 2, 3, 4, 5]
    biv_scan = [compute_transfer_entropy(X, Y, conditioning=None, lag=u, k=5, rng=rng) for u in lags]
    cond_scan = [compute_transfer_entropy(X, Y, conditioning=Z, lag=u, k=5, rng=rng) for u in lags]
    best_lag = lags[int(np.argmax(biv_scan))]
    te_biv, thr_biv, p_biv = surrogate_significance_test(X, Y, None, lag=best_lag, n_surrogates=25, seed=42)
    te_cond, thr_cond, p_cond = surrogate_significance_test(X, Y, Z, lag=best_lag, n_surrogates=25, seed=42)
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
    fig.patch.set_facecolor('#F8FAFC')
    ax1 = axes[0]
    ax1.set_facecolor('#F8FAFC')
    ax1.plot(lags, biv_scan, marker='o', linewidth=2.4, color='#E11D48', label='Bivariate TE: T(X -> Y)')
    ax1.plot(lags, cond_scan, marker='s', linewidth=2.4, color='#0D9488', label='Conditional TE: T(X -> Y | Z)')
    ax1.axhline(thr_cond, color='#64748B', linestyle='--', linewidth=1.2, label='95% Surrogate Null')
    ax1.set_xticks(lags)
    ax1.set_xlabel('Candidate Interaction Delay u (steps)', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Transfer Entropy (Bits)', fontsize=11, fontweight='bold')
    ax1.set_title('1. Interaction Delay Scan (' + motif_type + ')', fontsize=12, fontweight='bold')
    ax1.legend(frameon=True, facecolor='white')
    ax1.grid(True, linestyle='--', alpha=0.4)
    ax2 = axes[1]
    ax2.set_facecolor('#F8FAFC')
    labels = ['Bivariate TE (Lag ' + str(best_lag) + ')', 'Conditional TE | Z (Lag ' + str(best_lag) + ')']
    x_pos = np.arange(len(labels))
    ax2.bar(x_pos - 0.15, [te_biv, te_cond], width=0.3, color=['#E11D48', '#2563EB'], edgecolor='#0F172A', label='Estimated TE')
    ax2.bar(x_pos + 0.15, [thr_biv, thr_cond], width=0.3, color='#CBD5E1', edgecolor='#0F172A', label='95% Surrogate Cutoff')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(labels, fontweight='bold')
    ax2.set_ylabel('Transfer Entropy (Bits)', fontsize=11, fontweight='bold')
    ax2.set_title('2. Confounder & Cascade Filtering Test', fontsize=12, fontweight='bold')
    ax2.legend(frameon=True, facecolor='white')
    ax2.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()
    print(f'Motif: {motif_type} | Lag u={best_lag} | Bivariate TE={te_biv:.4f} (p={p_biv:.3f}) | Conditional TE={te_cond:.4f} (p={p_cond:.3f})')

widgets.interact(
    run_te_laboratory,
    motif_type=widgets.Dropdown(options=['Common_Driver', 'Indirect_Cascade', 'Direct_Coupling', 'Unobserved_Confounder'], value='Common_Driver', description='Causal Motif:'),
    coupling=widgets.FloatSlider(min=0.2, max=0.9, step=0.1, value=0.7, description='Coupling C:'),
    n_samples=widgets.IntSlider(min=250, max=800, step=50, value=450, description='Sample Size:'),
);
